# Diabetes 데이터셋 LassoCV 피처 선택

## 개요
- LassoCV를 활용한 임베디드 피처 선택
- Pipeline을 통한 효율적 모델 구축
- 교차검증 기반 최적 규제 강도 자동 선택

## 주요 단계
1. 데이터 로드 및 분리
2. Pipeline 구축 (Scaling → Selection → Regression)
3. LassoCV 기반 피처 선택
4. 모델 학습 및 평가

## 라이브러리 임포트

In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import LassoCV, Ridge
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score

## 1. 데이터 로드

**Diabetes 데이터셋**
- 442개 샘플, 10개 피처
- 당뇨병 진행도 예측

In [ ]:
# 데이터 로드
diabetes = load_diabetes()
X = pd.DataFrame(diabetes.data, columns=diabetes.feature_names)
y = diabetes.target

print(f"데이터 크기: {X.shape}")
print(f"피처 목록: {list(diabetes.feature_names)}")

**학습/테스트 데이터 분리**

In [ ]:
# 데이터 분리
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## 2. 문제 진단 및 해결 방법

**이전 방법의 문제점**
- 필터 방식 (f_regression): 단순 통계로 피처 간 상호작용 반영 못함
- 각 피처를 독립적으로 평가

**해결책: LassoCV 임베디드 방식**
- 모델 학습과 동시에 피처 선택
- L1 정규화로 불필요한 피처 계수를 0으로 만듦
- 교차검증으로 최적 규제 강도(alpha) 자동 선택
- 피처 간 상호작용 고려

## 3. Pipeline 구축

**Pipeline 구성**
1. StandardScaler: 피처 정규화
2. SelectFromModel + LassoCV: 임베디드 피처 선택
3. Ridge: 최종 예측 모델

**LassoCV 파라미터**
- cv=5: 5-Fold 교차검증
- random_state=42: 재현성 보장
- alpha: 자동으로 최적값 탐색

In [ ]:
# Pipeline 구축
pipe_lasso = Pipeline([
    ('scaler', StandardScaler()),
    ('feature_selection', SelectFromModel(LassoCV(cv=5, random_state=42))),
    ('regressor', Ridge(random_state=42))
])

print("Pipeline 구성 완료")
print(pipe_lasso)

## 4. 모델 학습

**학습 과정**
1. 데이터 스케일링
2. LassoCV로 최적 alpha 찾고 피처 선택
3. 선택된 피처로 Ridge 회귀 학습

In [ ]:
# 학습
print("Training Pipeline with LassoCV Selection...")
pipe_lasso.fit(X_train, y_train)
print("학습 완료!")

## 5. 선택된 피처 확인

**SelectFromModel 분석**
- LassoCV가 선택한 중요 피처 확인
- 계수가 0이 아닌 피처만 선택됨

In [ ]:
# 선택된 피처 확인
selector = pipe_lasso.named_steps['feature_selection']
selected_indices = selector.get_support(indices=True)
selected_features = np.array(diabetes.feature_names)[selected_indices]

print(f"선택된 피처 개수: {len(selected_features)}개 / {len(diabetes.feature_names)}개")
print(f"선택된 피처 목록: {list(selected_features)}")

**LassoCV 상세 정보**

In [ ]:
# LassoCV 모델 추출
lasso_model = selector.estimator_

print(f"최적 Alpha: {lasso_model.alpha_:.6f}")
print(f"\n각 피처별 계수:")
coef_df = pd.DataFrame({
    'Feature': diabetes.feature_names,
    'Coefficient': lasso_model.coef_,
    'Abs_Coef': np.abs(lasso_model.coef_),
    'Selected': selector.get_support()
}).sort_values('Abs_Coef', ascending=False)

print(coef_df)

## 6. 모델 평가

**R2 Score**
- 1에 가까울수록 좋은 성능
- 테스트 데이터로 일반화 성능 평가

In [ ]:
# 예측 및 평가
y_pred = pipe_lasso.predict(X_test)
r2 = r2_score(y_test, y_pred)

print(f"Pipeline R2 Score: {r2:.4f}")

## 7. 비교: 모든 피처 vs 선택된 피처

In [ ]:
# 모든 피처 사용 (피처 선택 없음)
pipe_all = Pipeline([
    ('scaler', StandardScaler()),
    ('regressor', Ridge(random_state=42))
])

pipe_all.fit(X_train, y_train)
y_pred_all = pipe_all.predict(X_test)
r2_all = r2_score(y_test, y_pred_all)

# 결과 비교
print("성능 비교:")
print(f"모든 피처 사용 (10개): R2 = {r2_all:.4f}")
print(f"선택된 피처 ({len(selected_features)}개): R2 = {r2:.4f}")
print(f"성능 차이: {r2 - r2_all:+.4f}")
print(f"\n피처 감소율: {(10 - len(selected_features)) / 10 * 100:.1f}%")

## 8. 예측 결과 시각화 (선택사항)

In [ ]:
import matplotlib.pyplot as plt

# 실제값 vs 예측값 시각화
plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred, alpha=0.6, edgecolors='k')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 
         'r--', lw=2, label='Perfect Prediction')
plt.xlabel('Actual Values')
plt.ylabel('Predicted Values')
plt.title(f'Actual vs Predicted (R2={r2:.4f})')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 정리

**LassoCV 임베디드 방식의 장점**
- 모델 학습과 피처 선택 동시 수행
- 피처 간 상호작용 고려
- 교차검증으로 과적합 방지
- 자동으로 최적 규제 강도 선택

**Pipeline의 이점**
- 데이터 누수 방지
- 전처리-선택-학습 일관성 보장
- 재현 가능한 모델 구축

**vs 필터 방식 (SelectKBest)**
- 필터: 빠르지만 피처 독립 평가
- LassoCV: 느리지만 피처 상호작용 고려

**추가 실험 아이디어**
- 다양한 alpha 범위 테스트
- ElasticNetCV 사용 (L1 + L2 정규화)
- 다른 회귀 모델과 성능 비교
- 교차검증 fold 수 변경 실험